# 1.1 Channel-space ERP similarity

For every iEEG electrode, compare its ERP with equally weighted means of MEG source ERPs in nonoverlapping MNI-distance shells, separately for every MEG participant. Trials are averaged within each condition, then the conditions receive equal weight. Source ERPs remain signed during averaging; the final Spearman correlations are absolute.

All computations run in `vs1_run_channel_space.py`, using functions in `vs1_1_channel_space.py`. Results and supporting counts are saved under `out/channel_space/`. Plotting code is editable below. This is a descriptive comparison of group-aligned ERPs, not a test with independent time samples or simultaneous recordings.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from nilearn import plotting
from IPython.display import display

ROOT = Path.cwd()
BLOCK_SCALING = True
CORRECT_IEEG_X1000 = True
CORRECT_MEG_ZSCORE = True
CONDITIONS = [1, 2]  # Mean within each condition, then equal-weight condition average
SHELL_WIDTH_MM = 5.
MAX_DISTANCE_MM = 100.
SHELL_EDGES_MM = np.r_[np.arange(0., MAX_DISTANCE_MM, SHELL_WIDTH_MM), MAX_DISTANCE_MM].tolist()
TIME_WINDOW = None  # Full epoch; alternatively [start_seconds, end_seconds]
ROI_COLUMN = 'region'
POOL_HEMISPHERES = True

sys.path[:0] = [str(ROOT), str(ROOT.parent), str(ROOT.parent / 'LB')]
from vs1_run_channel_space import prepare_analysis, result_directory
from vs1_coverage_figures import MODALITY_COLORS, apply_style
apply_style()


## Load saved analysis and configure inputs

Corrections follow the existing loader: iEEG ×1000 and MEG channel z-scoring before condition averaging, both switchable. Equal-variance scaling is applied to each participant's complete channel/source block after condition averaging, over the full epoch. Positive block scaling changes ERP amplitudes but leaves Spearman correlation unchanged.

Shells are ([r_mathrm{left},r_mathrm{right})); the final shell includes its upper boundary. Empty shells and constant ERPs have undefined correlations and are excluded from averages. Sources are not folded across hemispheres. ROI labels come from electrode metadata, with coordinate-based labels used if the selected column is absent. `POOL_HEMISPHERES` combines left/right ROI labels only.

Run the printed command, then reload this cell. `RECOMPUTE=True` prints a recomputation command and leaves plotting disabled until it is reset to `False`.


In [ ]:
INPUT_DIR = ROOT
OUTPUT_DIR = ROOT / 'out' / 'channel_space'
RECOMPUTE = False
MEG_DIR = INPUT_DIR / 'MEG' / 'dataMEG'
IEEG_DIR = INPUT_DIR / 'ieeg_shortWOBS_fs250'
METADATA_CSV = None
PROJECT_PATH = '/users/barbara/Desktop/MINDLAB2021_MEG/TempSeqAges/scratch/learning_bach_iEEG'
MEG_SUBJECTS = None  # None includes all available participants
IEEG_SUBJECTS = None
SFREQ = 250.
MEG_TMIN = None  # None uses the verified iEEG epoch start
MEG_TIMES_FILE = None

CONFIG = dict(output_dir=str(OUTPUT_DIR), block_scaling=BLOCK_SCALING,
    shell_edges_mm=SHELL_EDGES_MM, time_window=TIME_WINDOW,
    roi_column=ROI_COLUMN, pool_hemispheres=POOL_HEMISPHERES,
    data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
        metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
        project_path=str(PROJECT_PATH) if PROJECT_PATH else None,
        meg_subjects=MEG_SUBJECTS, ieeg_subjects=IEEG_SUBJECTS,
        meg_coordinate_unit='m', ieeg_coordinate_unit='mm',
        meg_scaling='channel_zscore' if CORRECT_MEG_ZSCORE else 'none',
        ieeg_scaling='none', ieeg_multiplier=1000. if CORRECT_IEEG_X1000 else 1.,
        meg_tmin=MEG_TMIN, meg_times_file=str(MEG_TIMES_FILE) if MEG_TIMES_FILE else None,
        sfreq=SFREQ, conditions=CONDITIONS))
results = prepare_analysis(CONFIG, OUTPUT_DIR / 'config.json', RECOMPUTE)
if results is not None:
    display(results['metadata'].groupby('subject').size().rename('n_electrodes'))
    display(results['shell_table'])
    display(results['electrode_summary'][['electrode_row','shell','n_meg','n_meg_with_sources','mean_n_sources']])
    print('Saved results:', result_directory(CONFIG))


## Example ERPs and electrode location

Select an iEEG participant and electrode, then display one MEG group-mean ERP per shell. The MEG mean gives equal weight to participants with sources in that shell. Bands are pointwise SEM across those MEG participants. Individual correlations are computed **before** averaging across MEG participants; the displayed mean waveform can cancel across signed sources or participants and need not have the same correlation as the mean absolute correlation.


In [ ]:
EXAMPLE_SUBJECT = None  # None selects the first saved iEEG participant
EXAMPLE_CHANNEL_INDEX = 0  # Zero-based channel_index in the saved epochs
EXAMPLE_SHELLS = [0, 1, 2, 3, 4, 5, 6]  # Zero-based shell indices
SHOW_ERP_SEM = True
FIGSIZE = (17, 6)
import numpy as np
if results is not None:
    metadata = results['metadata']
    subject = str(EXAMPLE_SUBJECT) if EXAMPLE_SUBJECT is not None else str(metadata.subject.iloc[0])
    electrodes = metadata.loc[metadata.subject.astype(str).eq(subject)]
    selected = electrodes.loc[electrodes.channel_index.eq(EXAMPLE_CHANNEL_INDEX)].iloc[0]
    row = int(selected.electrode_row)
    arrays = results['arrays']; times = arrays['times']
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
    brain = plotting.plot_glass_brain(None, figure=fig, axes=axes[0], display_mode='ortho', title=f'iEEG {subject} · {selected.channel}')
    brain.add_markers(electrodes[['x','y','z']].to_numpy(), marker_color='lightgray', marker_size=8)
    brain.add_markers(np.array([[selected.x,selected.y,selected.z]]), marker_color=MODALITY_COLORS['iEEG'], marker_size=65)
    axes[1].plot(times, arrays['ieeg_erp'][row], color=MODALITY_COLORS['iEEG'], linewidth=2.5, label=f'iEEG {selected.channel}')
    ax_t = axes[1].twinx()
    for index, shell in enumerate(EXAMPLE_SHELLS):
        bounds = results['shell_table'].set_index('shell').loc[shell]
        mean = arrays['meg_erp_mean'][row, shell]
        sem = arrays['meg_erp_sem'][row, shell]
        n = int(arrays['n_meg_erp'][row, shell])
        color = plt.get_cmap('autumn')(.1 + .8*index/max(1,len(EXAMPLE_SHELLS)-1))
        ax_t.plot(times, mean, color=color, label=f'MEG {bounds.left_mm:g}–{bounds.right_mm:g} mm (n={n})')
        if SHOW_ERP_SEM:
            ax_t.fill_between(times, mean-sem, mean+sem, color=color, alpha=.15)
    ax_t.set(xlabel='Time (s)', ylabel='ERP amplitude (after configured corrections/scaling)', title='Mean ERPs by source-distance shell')
    ax_t.grid(alpha=.2); 
    axes[1].legend(fontsize=8)
    ax_t.legend()
    plt.show()


## Electrode-level correlations for each iEEG participant

Each line is one electrode. Means and SEM are calculated across MEG participants with finite shell correlations. The brain marker and line share the same electrode color. Missing shells remain gaps; `n_meg` in the saved summary records the support for each point.


In [ ]:
SUBJECTS_TO_PLOT = None  # None plots every iEEG participant; otherwise use a list of IDs
SHOW_CHANNEL_LEGEND = False
CHANNEL_CMAP = 'turbo'
FIGSIZE = (18, 6)

if results is not None:
    metadata = results['metadata']
    subjects = list(metadata.subject.astype(str).unique()) if SUBJECTS_TO_PLOT is None else list(map(str, SUBJECTS_TO_PLOT))
    for subject in subjects:
        electrodes = metadata.loc[metadata.subject.astype(str).eq(subject)]
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE) 
        brain = plotting.plot_glass_brain(None, figure=fig, axes=axes[0], display_mode='ortho', title=f'iEEG {subject}')
        for index, electrode in enumerate(electrodes.itertuples()):
            color = to_hex(plt.get_cmap(CHANNEL_CMAP)(index/max(1,len(electrodes)-1)))
            frame = results['electrode_summary'].query('electrode_row == @electrode.electrode_row').sort_values('shell')
            if np.mean(frame.mean_abs_r) >0.2 :
                brain.add_markers(np.array([[electrode.x,electrode.y,electrode.z]]), marker_color=color, marker_size=15)
                axes[1].plot(frame.distance_mm, frame.mean_abs_r, 'o-', color=color, markersize=3, label=str(electrode.channel))
                axes[1].fill_between(frame.distance_mm, frame.mean_abs_r-frame['sem'], frame.mean_abs_r+frame['sem'], color=color, alpha=.1)
            else :
                brain.add_markers(np.array([[electrode.x,electrode.y,electrode.z]]), marker_color='grey', marker_size=15)

        axes[1].set(xlabel='Distance from electrode (mm; shell centers)', ylabel='Mean absolute Spearman correlation ± SEM', title='SEM across MEG participants')
        axes[1].grid(alpha=.2)
        if SHOW_CHANNEL_LEGEND:
            axes[1].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.02,1), ncol=2)
        plt.show()


## ROI-level correlations

Within each iEEG participant and ROI, average the electrode-level mean correlations. Then average these participant-level ROI values, giving each iEEG participant equal weight. Bands show SEM across contributing iEEG participants; a single contributing participant has no estimable SEM. These bands describe one sampling axis and do not incorporate joint uncertainty over both participant cohorts.


In [ ]:
ROIS_TO_PLOT = None  # None includes all available ROI labels
ROI_CMAP = 'tab20'
FIGSIZE = (13, 6)

if results is not None:
    table = results['roi_summary']
    regions = list(table.region.unique()) if ROIS_TO_PLOT is None else ROIS_TO_PLOT
    fig, ax = plt.subplots(figsize=FIGSIZE, layout='constrained')
    for index, region in enumerate(regions):
        frame = table.loc[table.region.eq(region)].sort_values('shell')
        color = plt.get_cmap(ROI_CMAP)(index/max(1,len(regions)-1))
        ax.plot(frame.distance_mm, frame.mean_abs_r, 'o-', color=color, label=region, markersize=4)
        ax.fill_between(frame.distance_mm, frame.mean_abs_r-frame['sem'], frame.mean_abs_r+frame['sem'], color=color, alpha=.15)
    ax.set(xlabel='Distance from electrode (mm; shell centers)', ylabel='Mean absolute Spearman correlation ± SEM', title='ROI means · SEM across iEEG participants')
    ax.grid(alpha=.2); ax.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=8)
    plt.show()


## Electrode markers with correlation halos

Select a participant and one shell. Each small marker locates an electrode; its larger halo encodes mean absolute correlation across MEG participants. Both halo color and size increase with correlation. Gray crosses indicate undefined correlations. This is a point display, not a spatial interpolation or a source-extent estimate.


In [ ]:
HALO_SUBJECT = metadata.subject.iloc[1]
HALO_SHELL = 13
HALO_MIN_SIZE = 35
HALO_MAX_SIZE = 350
FIGSIZE = (13, 6)

if results is not None:
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    metadata = results['metadata']
    subject = str(HALO_SUBJECT) if HALO_SUBJECT is not None else str(metadata.subject.iloc[0])
    electrodes = metadata.loc[metadata.subject.astype(str).eq(subject)]
    values = results['electrode_summary'].query('shell == @HALO_SHELL').set_index('electrode_row').mean_abs_r
    bounds = results['shell_table'].set_index('shell').loc[HALO_SHELL]
    fig = plt.figure(figsize=FIGSIZE)
    brain = plotting.plot_glass_brain(None, figure=fig, display_mode='ortho',
        title=f'iEEG {subject} · {bounds.left_mm:g}–{bounds.right_mm:g} mm · mean absolute correlation')
    for electrode in electrodes.itertuples():
        value = values.loc[electrode.electrode_row]
        xyz = np.array([[electrode.x,electrode.y,electrode.z]])
        if np.isfinite(value):
            brain.add_markers(xyz, marker_color=to_hex(plt.get_cmap('Reds')(value)),
                marker_size=HALO_MIN_SIZE+(HALO_MAX_SIZE-HALO_MIN_SIZE)*value, alpha=.5)
            brain.add_markers(xyz, marker_color='black', marker_size=5)
        else:
            brain.add_markers(xyz, marker_color='gray', marker='x', marker_size=20)
    color_axis = fig.add_axes([.92,.2,.015,.55])
    fig.colorbar(ScalarMappable(norm=Normalize(0,1), cmap='Reds'), cax=color_axis, label='Mean |Spearman r|')
    plt.show()


## ROI shell correlations at MEG source locations

One brain figure per ROI pools source locations from all included MEG participants. Each electrode-centered shell receives the ROI's mean absolute correlation from the analysis above: equal weight to MEG participants within each electrode, electrodes within each iEEG participant, then iEEG participants. At locations covered by multiple shells, average their values within each iEEG participant and then across participants. Identical MEG coordinates appear once; different coordinates remain separate, without interpolation or hemisphere folding.

Colors show **projected shell means**, not independently estimated source correlations. Black markers locate all iEEG electrodes in the selected ROI. Undefined locations are omitted. The saved spatial-support counts describe overlapping electrode shells, not independent observations. A common `Reds` scale of 0–1 permits comparison across ROIs. Run the printed command once, then reload the cell; the existing ERP analysis is reused.

### Voxel neighborhood averaging

The final cell averages the saved correlation values within `MEG_AVERAGING_RADIUS_MM` of each voxel center on an MNI grid with spacing `MEG_VOXEL_SIZE_MM`. Every finite, distinct source location in the sphere contributes equally; empty neighborhoods remain undefined. Change either parameter and rerun only the final cell. No ERP or correlation analysis is rerun, and no new cluster job is needed for smoothing. The smoothed images and contributing-source counts remain available in `meg_voxel_maps`.

The glass-brain display shows the maximum along each viewing axis of the spatially averaged volume. Neighborhoods use Euclidean distance without a cortical or hemisphere restriction; they can bridge nearby anatomical boundaries. These are descriptive smoothed shell maps, not new source-specific correlations.


In [ ]:
INPUT_DIR = result_directory(CONFIG)
OUTPUT_DIR = INPUT_DIR
RECOMPUTE_ROI_MAPS = False

import shlex
import json
ROI_MAP_FILE = OUTPUT_DIR / 'roi_shell_source_maps.csv'
ROI_MAP_CONFIG = OUTPUT_DIR / 'roi_shell_map_config.json'
roi_source_maps = None
if results is not None:
    ROI_MAP_CONFIG.write_text(json.dumps(CONFIG, indent=2))
    command = ['python', '-u', str(ROOT / 'vs1_run_channel_space.py'),
               '--config', str(ROI_MAP_CONFIG), '--roi-brain-maps']
    print('# Run on the cluster:')
    print(shlex.join(command))
    if ROI_MAP_FILE.exists() and not RECOMPUTE_ROI_MAPS:
        roi_source_maps = pd.read_csv(ROI_MAP_FILE)
    else:
        print('# Run the command, then set RECOMPUTE_ROI_MAPS=False and reload this cell.')


In [ ]:
BRAIN_ROIS_TO_PLOT = None  # None plots all ROIs; otherwise a list of saved ROI names
MEG_AVERAGING_RADIUS_MM = 5.  # Equal-weight spherical neighborhood, inclusive boundary
MEG_VOXEL_SIZE_MM = 2.  # MNI grid spacing; change and rerun this cell only
IEEG_MARKER_SIZE = 18
FIGSIZE = (15, 6)

if roi_source_maps is not None:
    import nibabel as nib
    from scipy.spatial import cKDTree
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    regions = list(roi_source_maps.region.unique()) if BRAIN_ROIS_TO_PLOT is None else BRAIN_ROIS_TO_PLOT
    norm = Normalize(0., 1.)
    cmap = plt.get_cmap('Reds')
    # Align every ROI to the same MNI grid. No recording data are reloaded.
    coordinates = roi_source_maps[['x', 'y', 'z']].to_numpy(float)
    origin = np.floor((coordinates.min(axis=0) - MEG_AVERAGING_RADIUS_MM) / MEG_VOXEL_SIZE_MM) * MEG_VOXEL_SIZE_MM
    upper = np.ceil((coordinates.max(axis=0) + MEG_AVERAGING_RADIUS_MM) / MEG_VOXEL_SIZE_MM) * MEG_VOXEL_SIZE_MM
    shape = tuple((np.rint((upper - origin) / MEG_VOXEL_SIZE_MM).astype(int) + 1).tolist())
    affine = np.diag([MEG_VOXEL_SIZE_MM] * 3 + [1.])
    affine[:3, 3] = origin
    meg_voxel_maps = {}  # In-memory images and source counts, replaced on each rerun
    for region in regions:
        sources = roi_source_maps.loc[roi_source_maps.region.eq(region)].dropna(subset=['mean_abs_r'])
        electrodes = results['metadata'].loc[lambda f: f.region_analysis.eq(region)]
        means = np.full(np.prod(shape), np.nan, dtype=np.float32)
        counts = np.zeros(np.prod(shape), dtype=np.int32)
        if len(sources):
            # Saved maps already contain one row per distinct source coordinate.
            tree = cKDTree(sources[['x', 'y', 'z']].to_numpy(float))
            values = sources.mean_abs_r.to_numpy(float)
            # Query in chunks to avoid storing all voxel neighborhoods at once.
            for start in range(0, len(means), 8192):
                stop = min(start + 8192, len(means))
                indices = np.column_stack(np.unravel_index(np.arange(start, stop), shape))
                centers = origin + indices * MEG_VOXEL_SIZE_MM
                neighbors = tree.query_ball_point(centers, np.nextafter(MEG_AVERAGING_RADIUS_MM, np.inf))
                for offset, rows in enumerate(neighbors):
                    counts[start + offset] = len(rows)
                    if rows:
                        means[start + offset] = values[rows].mean()
        # Keep unsupported voxels as NaN in the result; display them as background.
        image = nib.Nifti1Image(means.reshape(shape), affine)
        meg_voxel_maps[region] = dict(image=image, n_sources=counts.reshape(shape))
        display_image = nib.Nifti1Image(np.nan_to_num(means).reshape(shape), affine)
        fig = plt.figure(figsize=FIGSIZE)
        ax = fig.add_axes([.01, .1, .86, .8])
        brain = plotting.plot_glass_brain(display_image, figure=fig, axes=ax,
            display_mode='ortho', cmap='Reds', vmin=0., vmax=1., threshold=0.,
            colorbar=False, plot_abs=True,
            title=f'{region} · mean within {MEG_AVERAGING_RADIUS_MM:g} mm')
        brain.add_markers(electrodes[['x', 'y', 'z']].to_numpy(),
            marker_color='black', marker_size=IEEG_MARKER_SIZE)
        color_axis = fig.add_axes([.91, .22, .015, .55])
        fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=color_axis,
            label='Spatial mean of projected ROI shell |Spearman r|')
        fig.legend(handles=[Line2D([], [], linestyle='', marker='o', color='black',
                   label='iEEG electrodes')], loc='lower center', frameon=False)
        ax.grid(alpha=.2)
        plt.show()
